In [ ]:
import openeo
import time
from openeo.processes import quantiles, array_element, if_
import geopandas as gpd
import rasterio
from pathlib import Path
import numpy as np
import os
from scipy.ndimage import median_filter
from rasterio.mask import mask

In [ ]:
# Remove connection to the backend and remove the refresh token 
#from openeo.rest.auth.config import RefreshTokenStore
#RefreshTokenStore().remove()

In [ ]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

## Sentinel-2 mosaic from Sentinel-2 L2A

Create the Sentinel-2 mosaic from L2A data. Product description: 

https://dataspace.copernicus.eu/news/2024-2-27-exploring-new-frontier-sentinel-cloudless-mosaics-copernicus-data-space-ecosystem

https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-3-quarterly-mosaics

In [ ]:
spatial_extent_west = {"west": 610950, "east": 674650, "south": 5143820, "north": 5206380, "crs": "EPSG:32632"} # Western South Tyrol glaciers
spatial_extent_east = {"west": 704100, "east": 747040, "south": 5195880, "north": 5219660, "crs": "EPSG:32632"} # Eastern South Tyrol glaciers

In [ ]:
temporal_extent = ["2023-07-01T00:00:00Z", "2023-09-30T23:59:59Z"]

In [ ]:
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests")
year = 2023

In [ ]:
results_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/openEO/S2/" + str(year))
results_folder.mkdir(exist_ok=True)

### 1) Generate the Sentinel-2 L2A mosaics and indices 

In [ ]:
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
BAD_SCL = [1, 3, 7, 8, 9, 10]   # saturated, shadow, cloud (low/med/high), cirrus

def load_s2(spatial_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=BANDS + ["SCL"],
        max_cloud_cover=50,
    )
    return cube.resample_spatial(projection=32632, resolution=10, method="bilinear")

def cloud_mask(cube):
    scl = cube.band("SCL")
    m = scl == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (scl == v)
    return m

def create_s2_mosaic(spatial_extent):
    cube = load_s2(spatial_extent)
    mask = cloud_mask(cube)
    return (
        cube.mask(mask)
        .filter_bands(BANDS)
        .reduce_dimension(
            dimension="t",
            reducer=lambda x: array_element(quantiles(x, probabilities=[0.25]), index=0),
        )
    )

def glacier_mask(mosaic, ndsi_thr=0.4, ratio_thr=0.2):
    def classify(b):
        b03, b04, b11 = b["B03"], b["B04"], b["B11"]
        ndsi = (b03 - b11) / (b03 + b11)     # green vs SWIR
        ratio = (b04 - b11) / (b04 + b11)    # red vs SWIR
        return if_(ndsi > ndsi_thr, if_(ratio > ratio_thr, 1, 0), 0)
    return mosaic.reduce_dimension(dimension="bands", reducer=classify)

regions = {"west": spatial_extent_west, "east": spatial_extent_east}
jobs = {}

for name, extent in regions.items():
    print(f"Starting {name}...")
    result = glacier_mask(create_s2_mosaic(extent)).save_result(format="GTiff")
    job = result.create_job(title=f"S2_glacier_{name}_{year}")
    job.start()
    jobs[name] = job
    print(f"{name} started: {job.job_id}")
    time.sleep(30)

### 2) Load jobs

In [ ]:
from datetime import datetime
job_create_date = datetime.strptime('2026-10-05T00:00:00Z', "%Y-%m-%dT%H:%M:%SZ")
job_list = [connection.job(j['id']) for j in connection.list_jobs() if datetime.strptime(j['created'], "%Y-%m-%dT%H:%M:%SZ") > job_create_date]
#job_list = job_list[1:]
print(len(job_list), "jobs created after", job_create_date)
job_list

### 3) Download results

In [ ]:
for i, j in enumerate(job_list):

    print(i, j.describe()["title"], j.status())

    if j.status() == "finished":

        title = j.describe()["title"]
        filepath = os.path.join(results_folder, f"{title}.tiff")

        print(f"Downloading: {filepath}")

        j.get_results().download_file(filepath)

        print("Done!")